# Light Field Microscopy 3D Reconstruction Using Richardson-Lucy Deconvolution

## Introduction and Problem Background

**Light Field Microscopy (LFM)** is a revolutionary imaging technique that enables single-shot 3D volumetric imaging of biological specimens. Unlike conventional microscopy that captures a single focal plane, LFM uses a microlens array placed at the native image plane to simultaneously capture both spatial and angular information about the light field. This allows computational reconstruction of the entire 3D volume from a single 2D camera exposure.

### Why is this an Inverse Problem?

The fundamental challenge in LFM is that we observe a 2D projection (the lenslet image) of a 3D fluorescent volume. The forward imaging process **loses information** through:
1. **Dimensional reduction**: 3D → 2D projection
2. **Optical blurring**: Each depth plane has a different point spread function (PSF)
3. **Spatial-angular trade-off**: Resolution is traded for depth information
4. **Noise corruption**: Photon shot noise and detector noise

Recovering the 3D volume from the 2D measurement is therefore an **ill-posed inverse problem** — multiple 3D volumes could produce similar 2D projections, and noise amplification is a significant concern.

### Historical Context and Applications

Light field microscopy was pioneered by Levoy et al. (2006) and has since become a powerful tool for:
- **Neuroscience**: Whole-brain calcium imaging in zebrafish and C. elegans
- **Developmental biology**: Tracking cell dynamics in embryos
- **Cardiac imaging**: Studying heart function in small organisms

The Richardson-Lucy (RL) algorithm, originally developed independently by Richardson (1972) and Lucy (1974) for astronomical image restoration, has become the standard approach for LFM reconstruction due to its ability to handle Poisson noise statistics inherent to photon-limited imaging.

### Key References
1. Levoy, M., et al. "Light field microscopy." ACM SIGGRAPH (2006)
2. Broxton, M., et al. "Wave optics theory and 3-D deconvolution for the light field microscope." Optics Express (2013)
3. Richardson, W.H. "Bayesian-based iterative method of image restoration." JOSA (1972)
4. Lucy, L.B. "An iterative technique for the rectification of observed distributions." The Astronomical Journal (1974)

## Mathematical Formulation

### The Forward Model

In light field microscopy, the relationship between the 3D fluorescent volume and the 2D lenslet image can be expressed as a linear forward projection. Let $\mathbf{v} \in \mathbb{R}^{N_x \times N_y \times N_z}$ denote the 3D volume and $\mathbf{g} \in \mathbb{R}^{M_x \times M_y}$ denote the observed lenslet image.

**Equation 1: Forward Model**
$$\mathbf{g} = \mathbf{H} \mathbf{v} + \mathbf{n}$$

where:
- $\mathbf{H}$ is the forward projection operator (system matrix) that encodes how each voxel contributes to the lenslet image
- $\mathbf{n}$ represents measurement noise (primarily Poisson-distributed photon noise)

The forward operator $\mathbf{H}$ can be decomposed as a sum over depth planes:

**Equation 2: Depth-wise Forward Projection**
$$\mathbf{g} = \sum_{z=1}^{N_z} \mathbf{H}_z \mathbf{v}_z$$

where $\mathbf{H}_z$ represents the projection operator for depth plane $z$, encoding the depth-dependent point spread function and the lenslet geometry.

### The Inverse Problem

We seek to recover $\mathbf{v}$ given the observation $\mathbf{g}$. Under Poisson noise statistics, the maximum likelihood estimate is obtained by minimizing the negative log-likelihood:

**Equation 3: Poisson Negative Log-Likelihood**
$$\mathcal{L}(\mathbf{v}) = \sum_i \left[ (\mathbf{H}\mathbf{v})_i - g_i \log(\mathbf{H}\mathbf{v})_i \right]$$

### Richardson-Lucy Algorithm

The Richardson-Lucy algorithm provides an iterative solution that naturally enforces non-negativity and is well-suited for Poisson noise. The update rule is derived from the expectation-maximization (EM) framework:

**Equation 4: Richardson-Lucy Update Rule**
$$\mathbf{v}^{(k+1)} = \mathbf{v}^{(k)} \cdot \frac{\mathbf{H}^T \left( \frac{\mathbf{g}}{\mathbf{H}\mathbf{v}^{(k)}} \right)}{\mathbf{H}^T \mathbf{1}}$$

where:
- $\mathbf{H}^T$ is the backward projection (adjoint) operator
- $\mathbf{1}$ is a vector of ones
- The division and multiplication are element-wise

### Anti-Aliasing Regularization

To prevent high-frequency artifacts and improve reconstruction quality, we apply a depth-adaptive Lanczos filter after each iteration:

**Equation 5: Lanczos-Filtered Update**
$$\mathbf{v}_z^{(k+1)} = \left| \mathcal{F}^{-1} \left[ \mathbf{K}_z \cdot \mathcal{F}\left[ \tilde{\mathbf{v}}_z^{(k+1)} \right] \right] \right|$$

where $\mathbf{K}_z$ is the depth-dependent Lanczos kernel in Fourier space, and $\tilde{\mathbf{v}}_z^{(k+1)}$ is the unfiltered update.

### Convergence Metric

We monitor convergence using the mean absolute error between the forward projection and the observation:

**Equation 6: Convergence Metric**
$$\epsilon^{(k)} = \frac{1}{|\Omega|} \sum_{i \in \Omega} \left| \frac{g_i}{(\mathbf{H}\mathbf{v}^{(k)})_i} \cdot (\mathbf{H}\mathbf{1})_i - (\mathbf{H}\mathbf{1})_i \right|$$

This metric approaches zero as the reconstruction converges to a solution consistent with the observed data.

In [ ]:
# =============================================================================
# Cell 3: Environment Setup & Imports
# =============================================================================

import numpy as np
from scipy import ndimage
from scipy.fft import fft2, ifft2, fftshift, ifftshift
import matplotlib.pyplot as plt
from time import time
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib for publication-quality figures
plt.rcParams.update({
    'figure.figsize': (12, 8),
    'figure.dpi': 100,
    'font.size': 11,
    'font.family': 'serif',
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'image.cmap': 'viridis',
    'axes.grid': False
})

# Print library versions
print("Library Versions:")
print(f"  NumPy: {np.__version__}")
print(f"  SciPy: {ndimage.__name__} (from scipy)")
print(f"  Matplotlib: {plt.matplotlib.__version__}")
print("\nEnvironment setup complete.")

## Forward Model Explanation

### Physics of Light Field Microscopy

In a light field microscope, a microlens array (MLA) is placed at the native image plane of the microscope objective. Each microlens captures a different angular perspective of the sample, creating a pattern of sub-images (one per lenslet) on the camera sensor.

### Key Physical Parameters

1. **Lenslet pitch** ($p$): The spacing between adjacent microlenses (typically 100-150 μm)
2. **Lenslet focal length** ($f_{ML}$): Determines the angular sampling
3. **Objective magnification** ($M$): Relates object space to image space
4. **Depth range** ($[z_{min}, z_{max}]$): The axial extent being reconstructed
5. **Super-resolution factor** ($s$): Lateral upsampling for improved resolution

### The Projection Operator

For each depth plane $z$, the forward projection involves:

1. **Depth-dependent PSF**: Objects at different depths produce different blur patterns under each lenslet
2. **Geometric projection**: The position under each lenslet depends on the object's 3D location
3. **Intensity integration**: Contributions from all voxels are summed

The forward operator for a single depth plane can be written as:

$$\mathbf{H}_z = \mathbf{P}_z \cdot \mathbf{B}_z$$

where $\mathbf{P}_z$ is the geometric projection matrix and $\mathbf{B}_z$ is the depth-dependent blur operator.

### Simplified Forward Model for This Tutorial

For pedagogical clarity, we implement a simplified but physically meaningful forward model:

$$g(x,y) = \sum_{z} \left[ v(x,y,z) \ast h_z(x,y) \right] \cdot L(x,y)$$

where:
- $h_z(x,y)$ is the depth-dependent PSF (Gaussian with depth-varying width)
- $L(x,y)$ is the lenslet sampling pattern
- $\ast$ denotes convolution

This captures the essential physics: depth-dependent blurring and lenslet-based spatial sampling.

In [ ]:
# =============================================================================
# Cell 5: Forward Model & Synthetic Data Generation
# =============================================================================

def create_lenslet_pattern(shape, lenslet_pitch, offset=(0, 0)):
    """
    Create a binary lenslet sampling pattern.
    
    Args:
        shape: (height, width) of the pattern
        lenslet_pitch: spacing between lenslet centers in pixels
        offset: (y, x) offset for lenslet grid
    
    Returns:
        Binary mask indicating lenslet center regions
    """
    y, x = np.ogrid[:shape[0], :shape[1]]
    # Create periodic pattern
    y_mod = (y - offset[0]) % lenslet_pitch
    x_mod = (x - offset[1]) % lenslet_pitch
    # Each lenslet covers a region around its center
    radius = lenslet_pitch // 2 - 1
    center = lenslet_pitch // 2
    pattern = ((y_mod - center)**2 + (x_mod - center)**2) <= radius**2
    return pattern.astype(np.float32)


def create_depth_psf(shape, sigma):
    """
    Create a Gaussian PSF for a given depth (characterized by sigma).
    
    Args:
        shape: (height, width) of the PSF
        sigma: standard deviation of the Gaussian
    
    Returns:
        Normalized PSF
    """
    y, x = np.ogrid[:shape[0], :shape[1]]
    cy, cx = shape[0] // 2, shape[1] // 2
    psf = np.exp(-((y - cy)**2 + (x - cx)**2) / (2 * sigma**2))
    return psf / psf.sum()


def forward_project(volume, psf_sigmas, lenslet_pattern):
    """
    Forward projection: 3D volume -> 2D lenslet image.
    
    Implements: g = sum_z [ (v_z * h_z) . L ]
    
    Args:
        volume: 3D array (ny, nx, nz)
        psf_sigmas: list of PSF sigmas for each depth
        lenslet_pattern: 2D lenslet sampling pattern
    
    Returns:
        2D lenslet image
    """
    ny, nx, nz = volume.shape
    lf_image = np.zeros((ny, nx), dtype=np.float32)
    
    for z in range(nz):
        # Apply depth-dependent PSF via convolution
        psf = create_depth_psf((ny, nx), psf_sigmas[z])
        blurred = np.real(ifft2(fft2(volume[:, :, z]) * fft2(ifftshift(psf))))
        # Accumulate with lenslet weighting
        lf_image += blurred * lenslet_pattern
    
    return lf_image


def backward_project(lf_image, psf_sigmas, lenslet_pattern, nz):
    """
    Backward projection (adjoint): 2D lenslet image -> 3D volume.
    
    Implements: v = H^T g
    
    Args:
        lf_image: 2D lenslet image
        psf_sigmas: list of PSF sigmas for each depth
        lenslet_pattern: 2D lenslet sampling pattern
        nz: number of depth planes
    
    Returns:
        3D volume
    """
    ny, nx = lf_image.shape
    volume = np.zeros((ny, nx, nz), dtype=np.float32)
    
    # Apply lenslet pattern
    weighted_lf = lf_image * lenslet_pattern
    
    for z in range(nz):
        # Apply transposed (same for symmetric) PSF
        psf = create_depth_psf((ny, nx), psf_sigmas[z])
        # Correlation = convolution with flipped kernel (same for symmetric PSF)
        volume[:, :, z] = np.real(ifft2(fft2(weighted_lf) * np.conj(fft2(ifftshift(psf)))))
    
    return volume


def create_synthetic_volume(shape, n_objects=15):
    """
    Create a synthetic 3D fluorescent volume with point-like and extended objects.
    
    Args:
        shape: (ny, nx, nz) volume dimensions
        n_objects: number of fluorescent objects
    
    Returns:
        3D volume with synthetic fluorescent structures
    """
    ny, nx, nz = shape
    volume = np.zeros(shape, dtype=np.float32)
    
    # Add point sources at random locations
    for _ in range(n_objects):
        y = np.random.randint(ny // 4, 3 * ny // 4)
        x = np.random.randint(nx // 4, 3 * nx // 4)
        z = np.random.randint(0, nz)
        intensity = np.random.uniform(0.5, 1.0)
        
        # Create small Gaussian blob
        sigma_xy = np.random.uniform(2, 5)
        sigma_z = np.random.uniform(1, 2)
        
        yy, xx, zz = np.ogrid[:ny, :nx, :nz]
        blob = intensity * np.exp(-((yy - y)**2 + (xx - x)**2) / (2 * sigma_xy**2) 
                                   - (zz - z)**2 / (2 * sigma_z**2))
        volume += blob
    
    # Add some extended structures (filaments)
    for _ in range(3):
        z_plane = np.random.randint(0, nz)
        y_start = np.random.randint(ny // 4, ny // 2)
        x_start = np.random.randint(nx // 4, nx // 2)
        length = np.random.randint(20, 40)
        angle = np.random.uniform(0, 2 * np.pi)
        
        for t in range(length):
            y = int(y_start + t * np.sin(angle))
            x = int(x_start + t * np.cos(angle))
            if 0 <= y < ny and 0 <= x < nx:
                volume[y-1:y+2, x-1:x+2, z_plane] += 0.3
    
    return np.clip(volume, 0, None)


# =============================================================================
# Generate Synthetic Data
# =============================================================================

# Volume and imaging parameters
VOLUME_SHAPE = (128, 128, 7)  # (ny, nx, nz)
LENSLET_PITCH = 16  # pixels
DEPTH_RANGE = np.linspace(-3, 3, VOLUME_SHAPE[2])  # normalized depth units

# PSF sigma varies with depth (defocus model)
BASE_SIGMA = 2.0
PSF_SIGMAS = [BASE_SIGMA + 0.8 * abs(d) for d in DEPTH_RANGE]

print("Simulation Parameters:")
print(f"  Volume shape: {VOLUME_SHAPE}")
print(f"  Number of depth planes: {VOLUME_SHAPE[2]}")
print(f"  Lenslet pitch: {LENSLET_PITCH} pixels")
print(f"  PSF sigmas: {[f'{s:.2f}' for s in PSF_SIGMAS]}")

# Create ground truth volume
print("\nGenerating synthetic ground truth volume...")
ground_truth_volume = create_synthetic_volume(VOLUME_SHAPE, n_objects=12)
print(f"  Volume range: [{ground_truth_volume.min():.3f}, {ground_truth_volume.max():.3f}]")

# Create lenslet pattern
lenslet_pattern = create_lenslet_pattern(VOLUME_SHAPE[:2], LENSLET_PITCH)
print(f"  Lenslet pattern coverage: {100 * lenslet_pattern.mean():.1f}%")

# Forward projection to create clean lenslet image
print("\nApplying forward model...")
clean_lf_image = forward_project(ground_truth_volume, PSF_SIGMAS, lenslet_pattern)

# Add Poisson noise (realistic for fluorescence microscopy)
PHOTON_LEVEL = 1000  # peak photon count
scaled_image = clean_lf_image * PHOTON_LEVEL / (clean_lf_image.max() + 1e-10)
noisy_lf_image = np.random.poisson(scaled_image).astype(np.float32)
noisy_lf_image = noisy_lf_image / PHOTON_LEVEL * clean_lf_image.max()

# Add small read noise
READ_NOISE_STD = 0.01 * clean_lf_image.max()
noisy_lf_image += np.random.normal(0, READ_NOISE_STD, noisy_lf_image.shape)
noisy_lf_image = np.clip(noisy_lf_image, 0, None)

print(f"  Clean LF image range: [{clean_lf_image.min():.3f}, {clean_lf_image.max():.3f}]")
print(f"  Noisy LF image range: [{noisy_lf_image.min():.3f}, {noisy_lf_image.max():.3f}]")

# Visualize the forward model results
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Ground truth slices
for i, z_idx in enumerate([0, VOLUME_SHAPE[2]//2, VOLUME_SHAPE[2]-1]):
    im = axes[0, i].imshow(ground_truth_volume[:, :, z_idx], cmap='hot')
    axes[0, i].set_title(f'Ground Truth (z={z_idx}, depth={DEPTH_RANGE[z_idx]:.1f})')
    axes[0, i].axis('off')
    plt.colorbar(im, ax=axes[0, i], fraction=0.046)

# Lenslet pattern and images
axes[1, 0].imshow(lenslet_pattern, cmap='gray')
axes[1, 0].set_title('Lenslet Sampling Pattern')
axes[1, 0].axis('off')

im1 = axes[1, 1].imshow(clean_lf_image, cmap='hot')
axes[1, 1].set_title('Clean Lenslet Image (Forward Projection)')
axes[1, 1].axis('off')
plt.colorbar(im1, ax=axes[1, 1], fraction=0.046)

im2 = axes[1, 2].imshow(noisy_lf_image, cmap='hot')
axes[1, 2].set_title(f'Noisy Lenslet Image (Photon level={PHOTON_LEVEL})')
axes[1, 2].axis('off')
plt.colorbar(im2, ax=axes[1, 2], fraction=0.046)

plt.suptitle('Forward Model: 3D Volume → 2D Lenslet Image', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nSynthetic data generation complete.")

## Reconstruction Algorithm: Richardson-Lucy Deconvolution

### Algorithm Overview

The Richardson-Lucy (RL) algorithm is an iterative expectation-maximization method that finds the maximum likelihood estimate under Poisson noise statistics. It has several desirable properties for light field microscopy:

1. **Non-negativity preservation**: If initialized with positive values, the reconstruction remains positive
2. **Flux conservation**: Total intensity is preserved
3. **Poisson-optimal**: Derived from the Poisson likelihood, matching photon noise statistics

### Step-by-Step Algorithm

**Initialization:**
- Set $\mathbf{v}^{(0)} = \mathbf{1}$ (uniform volume)
- Precompute $\mathbf{H}^T \mathbf{1}$ (normalization factor)

**Iteration (for $k = 0, 1, 2, ...$):**

1. **Forward projection**: Compute predicted measurement
   $$\hat{\mathbf{g}}^{(k)} = \mathbf{H} \mathbf{v}^{(k)}$$

2. **Ratio computation**: Element-wise division
   $$\mathbf{r}^{(k)} = \frac{\mathbf{g}}{\hat{\mathbf{g}}^{(k)}}$$

3. **Backward projection**: Propagate error back to volume space
   $$\mathbf{e}^{(k)} = \mathbf{H}^T \mathbf{r}^{(k)}$$

4. **Multiplicative update**: 
   $$\mathbf{v}^{(k+1)} = \mathbf{v}^{(k)} \cdot \frac{\mathbf{e}^{(k)}}{\mathbf{H}^T \mathbf{1}}$$

5. **Anti-aliasing filter** (optional): Apply Lanczos filter to suppress high-frequency artifacts

### Convergence Properties

- The RL algorithm is guaranteed to **monotonically decrease** the Poisson negative log-likelihood
- Convergence is typically slow (100+ iterations for good results)
- **Semi-convergence**: Due to noise, the reconstruction quality may degrade after too many iterations (regularization by early stopping)

### Hyperparameter Choices

| Parameter | Typical Value | Effect |
|-----------|---------------|--------|
| `n_iterations` | 50-200 | More iterations → sharper but potentially noisier |
| `filter_flag` | True | Anti-aliasing improves stability |
| `lanczos_window` | 3-5 | Larger → smoother filtering |

### Lanczos Anti-Aliasing Filter

The Lanczos filter is applied in Fourier space with a depth-dependent cutoff frequency:

$$K_z(f) = \text{sinc}\left(\frac{f}{f_c(z)}\right) \cdot \text{sinc}\left(\frac{f}{a \cdot f_c(z)}\right)$$

where $f_c(z)$ is the depth-dependent cutoff frequency and $a$ is the window parameter.

In [ ]:
# =============================================================================
# Cell 7: Reconstruction Implementation (Richardson-Lucy Algorithm)
# =============================================================================

def create_lanczos_kernel_fft(shape, cutoff_freqs, window_size=3):
    """
    Create depth-dependent Lanczos filter kernels in Fourier space.
    
    Args:
        shape: (ny, nx, nz) volume shape
        cutoff_freqs: list of cutoff frequencies for each depth
        window_size: Lanczos window parameter (typically 2-4)
    
    Returns:
        3D array of Fourier-space filter kernels
    """
    ny, nx, nz = shape
    kernels = np.zeros(shape, dtype=np.float32)
    
    # Frequency grids
    fy = np.fft.fftfreq(ny)
    fx = np.fft.fftfreq(nx)
    FY, FX = np.meshgrid(fy, fx, indexing='ij')
    F = np.sqrt(FX**2 + FY**2)
    
    for z in range(nz):
        fc = cutoff_freqs[z]
        # Lanczos kernel: sinc(f/fc) * sinc(f/(a*fc))
        with np.errstate(divide='ignore', invalid='ignore'):
            arg1 = np.pi * F / fc
            arg2 = np.pi * F / (window_size * fc)
            sinc1 = np.where(arg1 == 0, 1.0, np.sin(arg1) / arg1)
            sinc2 = np.where(arg2 == 0, 1.0, np.sin(arg2) / arg2)
        kernel = sinc1 * sinc2
        kernel[F > fc * window_size] = 0  # Hard cutoff
        kernels[:, :, z] = kernel
    
    return kernels


def richardson_lucy_lfm(observed_image, psf_sigmas, lenslet_pattern, volume_shape,
                        n_iterations=50, filter_flag=True, lanczos_window=3,
                        verbose=True):
    """
    Richardson-Lucy deconvolution for light field microscopy.
    
    Implements the iterative update:
        v^(k+1) = v^(k) * (H^T(g / Hv^(k))) / (H^T 1)
    
    Args:
        observed_image: 2D noisy lenslet image
        psf_sigmas: list of PSF sigmas for each depth
        lenslet_pattern: 2D lenslet sampling pattern
        volume_shape: (ny, nx, nz) output volume shape
        n_iterations: number of RL iterations
        filter_flag: whether to apply anti-aliasing filter
        lanczos_window: Lanczos filter window size
        verbose: print progress
    
    Returns:
        tuple: (reconstructed_volume, error_metrics, iteration_times)
    """
    ny, nx, nz = volume_shape
    
    # Initialize with uniform volume
    recon_volume = np.ones(volume_shape, dtype=np.float32)
    
    # Precompute normalization: H^T(1)
    if verbose:
        print("Precomputing normalization factors...")
    ones_forward = forward_project(np.ones(volume_shape, dtype=np.float32), 
                                   psf_sigmas, lenslet_pattern)
    ones_backward = backward_project(ones_forward, psf_sigmas, lenslet_pattern, nz)
    ones_backward = np.maximum(ones_backward, 1e-10)  # Avoid division by zero
    
    # Prepare anti-aliasing filter
    if filter_flag:
        # Cutoff frequencies decrease with defocus (larger PSF sigma)
        cutoff_freqs = [0.4 / (1 + 0.1 * s) for s in psf_sigmas]
        lanczos_kernels = create_lanczos_kernel_fft(volume_shape, cutoff_freqs, lanczos_window)
    
    # Tracking metrics
    error_metrics = []
    iteration_times = []
    
    # Small constant to avoid division by zero
    eps = 1e-10
    
    if verbose:
        print(f"\nStarting Richardson-Lucy reconstruction ({n_iterations} iterations)...")
    
    t_start = time()
    
    for iteration in range(n_iterations):
        t_iter_start = time()
        
        # Step 1: Forward projection
        predicted = forward_project(recon_volume, psf_sigmas, lenslet_pattern)
        predicted = np.maximum(predicted, eps)
        
        # Step 2: Compute ratio (error in measurement space)
        ratio = observed_image / predicted
        ratio = np.nan_to_num(ratio, nan=0.0, posinf=0.0, neginf=0.0)
        
        # Compute convergence metric
        error_lf = ratio * ones_forward
        error_metric = np.mean(np.abs(error_lf - ones_forward))
        error_metrics.append(error_metric)
        
        # Step 3: Backward projection of ratio
        correction = backward_project(ratio * ones_forward, psf_sigmas, lenslet_pattern, nz)
        
        # Step 4: Multiplicative update
        correction = correction / ones_backward
        correction = np.nan_to_num(correction, nan=1.0, posinf=1.0, neginf=1.0)
        recon_volume = recon_volume * correction
        
        # Step 5: Apply anti-aliasing filter (optional)
        if filter_flag:
            for z in range(nz):
                filtered = ifft2(lanczos_kernels[:, :, z] * fft2(recon_volume[:, :, z]))
                recon_volume[:, :, z] = np.abs(filtered)
        
        # Ensure non-negativity
        recon_volume = np.maximum(recon_volume, 0)
        recon_volume = np.nan_to_num(recon_volume, nan=0.0, posinf=0.0, neginf=0.0)
        
        t_iter_end = time()
        iteration_times.append(t_iter_end - t_iter_start)
        
        if verbose and (iteration + 1) % 10 == 0:
            print(f"  Iteration {iteration + 1:3d}/{n_iterations}: "
                  f"Error = {error_metric:.6f}, Time = {t_iter_end - t_iter_start:.3f}s")
    
    t_total = time() - t_start
    
    if verbose:
        print(f"\nReconstruction complete!")
        print(f"  Total time: {t_total:.2f} seconds")
        print(f"  Average time per iteration: {np.mean(iteration_times):.3f} seconds")
        print(f"  Final error metric: {error_metrics[-1]:.6f}")
    
    return recon_volume, error_metrics, iteration_times


# =============================================================================
# Run Reconstruction
# =============================================================================

N_ITERATIONS = 50
FILTER_FLAG = True
LANCZOS_WINDOW = 3

print("="*60)
print("RICHARDSON-LUCY RECONSTRUCTION")
print("="*60)
print(f"\nReconstruction parameters:")
print(f"  Number of iterations: {N_ITERATIONS}")
print(f"  Anti-aliasing filter: {FILTER_FLAG}")
print(f"  Lanczos window size: {LANCZOS_WINDOW}")

reconstructed_volume, error_metrics, iteration_times = richardson_lucy_lfm(
    observed_image=noisy_lf_image,
    psf_sigmas=PSF_SIGMAS,
    lenslet_pattern=lenslet_pattern,
    volume_shape=VOLUME_SHAPE,
    n_iterations=N_ITERATIONS,
    filter_flag=FILTER_FLAG,
    lanczos_window=LANCZOS_WINDOW,
    verbose=True
)

print(f"\nReconstructed volume shape: {reconstructed_volume.shape}")
print(f"Reconstructed volume range: [{reconstructed_volume.min():.4f}, {reconstructed_volume.max():.4f}]")

## Results Visualization

In this section, we will visualize the reconstruction results through several complementary views:

### What to Look For

1. **Spatial fidelity**: Do the reconstructed structures match the ground truth locations?
2. **Depth localization**: Are objects correctly placed in the z-dimension?
3. **Intensity recovery**: Is the relative brightness of objects preserved?
4. **Artifact levels**: Are there ringing artifacts, noise amplification, or missing structures?

### Visualization Strategy

We will display:
- **Side-by-side depth slices**: Ground truth vs reconstruction for each z-plane
- **Maximum intensity projections (MIP)**: XY, XZ, and YZ views
- **Quantitative metrics**: PSNR, SSIM, and normalized cross-correlation

### Expected Behavior

For well-posed reconstructions:
- Central depth planes (in-focus) should reconstruct better than peripheral planes
- Point sources should appear as localized spots, not elongated along z
- The reconstruction should be smoother than the ground truth due to the regularizing effect of the anti-aliasing filter

In [ ]:
# =============================================================================
# Cell 9: Visualization - Ground Truth vs Reconstruction
# =============================================================================

def compute_psnr(img_true, img_test, data_range=None):
    """Compute Peak Signal-to-Noise Ratio."""
    if data_range is None:
        data_range = img_true.max() - img_true.min()
    mse = np.mean((img_true - img_test) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(data_range ** 2 / mse)


def compute_ssim(img_true, img_test, win_size=7):
    """Compute Structural Similarity Index (simplified version)."""
    C1 = (0.01 * img_true.max()) ** 2
    C2 = (0.03 * img_true.max()) ** 2
    
    # Local means
    mu_true = ndimage.uniform_filter(img_true, win_size)
    mu_test = ndimage.uniform_filter(img_test, win_size)
    
    # Local variances and covariance
    sigma_true_sq = ndimage.uniform_filter(img_true ** 2, win_size) - mu_true ** 2
    sigma_test_sq = ndimage.uniform_filter(img_test ** 2, win_size) - mu_test ** 2
    sigma_true_test = ndimage.uniform_filter(img_true * img_test, win_size) - mu_true * mu_test
    
    # SSIM formula
    numerator = (2 * mu_true * mu_test + C1) * (2 * sigma_true_test + C2)
    denominator = (mu_true ** 2 + mu_test ** 2 + C1) * (sigma_true_sq + sigma_test_sq + C2)
    
    return np.mean(numerator / denominator)


def compute_ncc(img_true, img_test):
    """Compute Normalized Cross-Correlation."""
    img_true_norm = img_true - img_true.mean()
    img_test_norm = img_test - img_test.mean()
    return np.sum(img_true_norm * img_test_norm) / (
        np.sqrt(np.sum(img_true_norm ** 2) * np.sum(img_test_norm ** 2)) + 1e-10
    )


def normalize_for_display(img):
    """Normalize image to [0, 1] for display."""
    img_min, img_max = img.min(), img.max()
    if img_max - img_min < 1e-10:
        return np.zeros_like(img)
    return (img - img_min) / (img_max - img_min)


# =============================================================================
# Compute metrics for each depth slice
# =============================================================================

print("Computing quantitative metrics...\n")

slice_metrics = []
for z in range(VOLUME_SHAPE[2]):
    gt_slice = ground_truth_volume[:, :, z]
    recon_slice = reconstructed_volume[:, :, z]
    
    # Normalize for fair comparison
    gt_norm = normalize_for_display(gt_slice)
    recon_norm = normalize_for_display(recon_slice)
    
    psnr = compute_psnr(gt_norm, recon_norm, data_range=1.0)
    ssim = compute_ssim(gt_norm, recon_norm)
    ncc = compute_ncc(gt_slice, recon_slice)
    
    slice_metrics.append({'z': z, 'depth': DEPTH_RANGE[z], 'PSNR': psnr, 'SSIM': ssim, 'NCC': ncc})

# Overall volume metrics
gt_vol_norm = normalize_for_display(ground_truth_volume)
recon_vol_norm = normalize_for_display(reconstructed_volume)
overall_psnr = compute_psnr(gt_vol_norm, recon_vol_norm, data_range=1.0)
overall_ssim = compute_ssim(gt_vol_norm.mean(axis=2), recon_vol_norm.mean(axis=2))
overall_ncc = compute_ncc(ground_truth_volume, reconstructed_volume)

print(f"Overall Volume Metrics:")
print(f"  PSNR: {overall_psnr:.2f} dB")
print(f"  SSIM: {overall_ssim:.4f}")
print(f"  NCC:  {overall_ncc:.4f}")

# =============================================================================
# Visualization: Depth slices comparison
# =============================================================================

n_depths = VOLUME_SHAPE[2]
fig, axes = plt.subplots(3, n_depths, figsize=(2.5 * n_depths, 7))

for z in range(n_depths):
    # Ground truth
    im0 = axes[0, z].imshow(ground_truth_volume[:, :, z], cmap='hot', vmin=0)
    axes[0, z].set_title(f'z={z}\n(d={DEPTH_RANGE[z]:.1f})', fontsize=9)
    axes[0, z].axis('off')
    
    # Reconstruction
    im1 = axes[1, z].imshow(reconstructed_volume[:, :, z], cmap='hot', vmin=0)
    axes[1, z].set_title(f'PSNR={slice_metrics[z]["PSNR"]:.1f}dB', fontsize=9)
    axes[1, z].axis('off')
    
    # Difference (error)
    diff = np.abs(normalize_for_display(ground_truth_volume[:, :, z]) - 
                  normalize_for_display(reconstructed_volume[:, :, z]))
    im2 = axes[2, z].imshow(diff, cmap='coolwarm', vmin=0, vmax=0.5)
    axes[2, z].axis('off')

# Row labels
axes[0, 0].set_ylabel('Ground\nTruth', fontsize=11, rotation=0, ha='right', va='center')
axes[1, 0].set_ylabel('Recon-\nstruction', fontsize=11, rotation=0, ha='right', va='center')
axes[2, 0].set_ylabel('Absolute\nError', fontsize=11, rotation=0, ha='right', va='center')

plt.suptitle('Depth-wise Comparison: Ground Truth vs Richardson-Lucy Reconstruction', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# =============================================================================
# Maximum Intensity Projections
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Ground truth MIPs
axes[0, 0].imshow(ground_truth_volume.max(axis=2), cmap='hot')
axes[0, 0].set_title('Ground Truth - XY MIP')
axes[0, 0].set_xlabel('X')
axes[0, 0].set_ylabel('Y')

axes[0, 1].imshow(ground_truth_volume.max(axis=0), cmap='hot', aspect='auto')
axes[0, 1].set_title('Ground Truth - YZ MIP')
axes[0, 1].set_xlabel('Z')
axes[0, 1].set_ylabel('X')

axes[0, 2].imshow(ground_truth_volume.max(axis=1), cmap='hot', aspect='auto')
axes[0, 2].set_title('Ground Truth - XZ MIP')
axes[0, 2].set_xlabel('Z')
axes[0, 2].set_ylabel('Y')

# Reconstruction MIPs
axes[1, 0].imshow(reconstructed_volume.max(axis=2), cmap='hot')
axes[1, 0].set_title('Reconstruction - XY MIP')
axes[1, 0].set_xlabel('X')
axes[1, 0].set_ylabel('Y')

axes[1, 1].imshow(reconstructed_volume.max(axis=0), cmap='hot', aspect='auto')
axes[1, 1].set_title('Reconstruction - YZ MIP')
axes[1, 1].set_xlabel('Z')
axes[1, 1].set_ylabel('X')

axes[1, 2].imshow(reconstructed_volume.max(axis=1), cmap='hot', aspect='auto')
axes[1, 2].set_title('Reconstruction - XZ MIP')
axes[1, 2].set_xlabel('Z')
axes[1, 2].set_ylabel('Y')

plt.suptitle(f'Maximum Intensity Projections (Overall PSNR: {overall_psnr:.2f} dB, NCC: {overall_ncc:.4f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Convergence Analysis

### Expected Convergence Behavior

The Richardson-Lucy algorithm exhibits characteristic convergence properties:

1. **Rapid initial decrease**: The error metric drops quickly in the first 10-20 iterations as the algorithm captures the main features

2. **Gradual refinement**: Subsequent iterations refine details with diminishing returns

3. **Semi-convergence**: For noisy data, the reconstruction quality may actually degrade after too many iterations as the algorithm starts fitting noise

### Diagnosing Problems

| Symptom | Possible Cause | Solution |
|---------|----------------|----------|
| Error plateaus early | Regularization too strong | Reduce filter strength |
| Error oscillates | Numerical instability | Add small epsilon to divisions |
| Error increases | Overfitting noise | Use early stopping |
| Slow convergence | Poor initialization | Use better initial guess |

### Monitoring Metrics

We track the mean absolute error (MAE) between the forward projection of the current estimate and the observed data, normalized by the "ones forward" projection. This metric:
- Approaches zero for perfect reconstruction
- Is scale-invariant
- Directly relates to the multiplicative update factor

In [ ]:
# =============================================================================
# Cell 11: Convergence Curve Plot
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].plot(range(1, len(error_metrics) + 1), error_metrics, 'b-', linewidth=2, marker='o', 
             markersize=4, markevery=5, label='LF MAE')
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('Mean Absolute Error', fontsize=12)
axes[0].set_title('Convergence Curve (Linear Scale)', fontsize=13)
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[0].set_xlim([1, len(error_metrics)])

# Log scale
axes[1].semilogy(range(1, len(error_metrics) + 1), error_metrics, 'b-', linewidth=2, marker='o',
                 markersize=4, markevery=5, label='LF MAE')
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Mean Absolute Error (log scale)', fontsize=12)
axes[1].set_title('Convergence Curve (Log Scale)', fontsize=13)
axes[1].grid(True, alpha=0.3, which='both')
axes[1].legend()
axes[1].set_xlim([1, len(error_metrics)])

# Add annotations
initial_error = error_metrics[0]
final_error = error_metrics[-1]
reduction_factor = initial_error / final_error

axes[0].annotate(f'Initial: {initial_error:.4f}', xy=(1, initial_error), 
                 xytext=(5, initial_error * 1.1), fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='gray'))
axes[0].annotate(f'Final: {final_error:.4f}', xy=(len(error_metrics), final_error),
                 xytext=(len(error_metrics) - 10, final_error * 1.5), fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='gray'))

plt.suptitle(f'Richardson-Lucy Convergence ({reduction_factor:.1f}x error reduction over {N_ITERATIONS} iterations)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Iteration timing analysis
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, len(iteration_times) + 1), iteration_times, color='steelblue', alpha=0.7)
ax.axhline(y=np.mean(iteration_times), color='red', linestyle='--', linewidth=2, 
           label=f'Mean: {np.mean(iteration_times):.3f}s')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title('Computation Time per Iteration', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nConvergence Statistics:")
print(f"  Initial error: {initial_error:.6f}")
print(f"  Final error: {final_error:.6f}")
print(f"  Error reduction: {reduction_factor:.2f}x")
print(f"  Total computation time: {sum(iteration_times):.2f} seconds")
print(f"  Mean iteration time: {np.mean(iteration_times):.4f} seconds")

## Error Analysis & Sensitivity

### Sources of Error in LFM Reconstruction

1. **Model mismatch**: Our simplified forward model doesn't capture all optical effects (aberrations, vignetting, etc.)

2. **Noise amplification**: The inverse problem is ill-conditioned, meaning small noise in the measurement can cause large errors in the reconstruction

3. **Depth ambiguity**: Objects at different depths can produce similar lenslet patterns, leading to depth localization errors

4. **Spatial resolution loss**: The trade-off between angular and spatial sampling limits lateral resolution

### Regularization Effects

The anti-aliasing Lanczos filter acts as implicit regularization:
- **Benefit**: Suppresses high-frequency noise and ringing artifacts
- **Cost**: Reduces sharpness and may blur fine details

### Sensitivity Study

We will investigate how reconstruction quality depends on:
1. **Noise level**: Higher noise → worse reconstruction
2. **Number of iterations**: Trade-off between sharpness and noise
3. **Filter strength**: Trade-off between smoothness and detail

In [ ]:
# =============================================================================
# Cell 13: Error Map & Sensitivity Analysis
# =============================================================================

# =============================================================================
# Part 1: Detailed Error Analysis
# =============================================================================

# Compute error maps
error_volume = np.abs(normalize_for_display(ground_truth_volume) - 
                      normalize_for_display(reconstructed_volume))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Error maps for selected depth slices
selected_depths = [0, VOLUME_SHAPE[2]//4, VOLUME_SHAPE[2]//2, VOLUME_SHAPE[2]-1]
for i, z in enumerate(selected_depths):
    im = axes[0, i].imshow(error_volume[:, :, z], cmap='hot', vmin=0, vmax=0.5)
    axes[0, i].set_title(f'Error Map (z={z}, d={DEPTH_RANGE[z]:.1f})')
    axes[0, i].axis('off')
    plt.colorbar(im, ax=axes[0, i], fraction=0.046)

# Error statistics per depth
depth_errors = [error_volume[:, :, z].mean() for z in range(VOLUME_SHAPE[2])]
depth_stds = [error_volume[:, :, z].std() for z in range(VOLUME_SHAPE[2])]

axes[1, 0].bar(range(VOLUME_SHAPE[2]), depth_errors, yerr=depth_stds, 
               color='steelblue', alpha=0.7, capsize=3)
axes[1, 0].set_xlabel('Depth Index')
axes[1, 0].set_ylabel('Mean Absolute Error')
axes[1, 0].set_title('Error vs Depth')
axes[1, 0].grid(True, alpha=0.3)

# PSNR per depth
psnr_values = [m['PSNR'] for m in slice_metrics]
axes[1, 1].bar(range(VOLUME_SHAPE[2]), psnr_values, color='forestgreen', alpha=0.7)
axes[1, 1].set_xlabel('Depth Index')
axes[1, 1].set_ylabel('PSNR (dB)')
axes[1, 1].set_title('PSNR vs Depth')
axes[1, 1].grid(True, alpha=0.3)

# Error histogram
axes[1, 2].hist(error_volume.flatten(), bins=50, color='coral', alpha=0.7, edgecolor='black')
axes[1, 2].set_xlabel('Absolute Error')
axes[1, 2].set_ylabel('Frequency')
axes[1, 2].set_title('Error Distribution')
axes[1, 2].axvline(x=error_volume.mean(), color='red', linestyle='--', 
                   label=f'Mean: {error_volume.mean():.4f}')
axes[1, 2].legend()

# Scatter: GT intensity vs Reconstruction intensity
gt_flat = ground_truth_volume.flatten()
recon_flat = reconstructed_volume.flatten()
# Subsample for visualization
idx = np.random.choice(len(gt_flat), min(5000, len(gt_flat)), replace=False)
axes[1, 3].scatter(gt_flat[idx], recon_flat[idx], alpha=0.3, s=5, c='blue')
max_val = max(gt_flat.max(), recon_flat.max())
axes[1, 3].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Ideal')
axes[1, 3].set_xlabel('Ground Truth Intensity')
axes[1, 3].set_ylabel('Reconstructed Intensity')
axes[1, 3].set_title('Intensity Correlation')
axes[1, 3].legend()
axes[1, 3].set_aspect('equal')

plt.suptitle('Detailed Error Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# =============================================================================
# Part 2: Sensitivity to Noise Level
# =============================================================================

print("\n" + "="*60)
print("SENSITIVITY ANALYSIS: Effect of Noise Level")
print("="*60)

noise_levels = [100, 500, 1000, 5000]  # Photon counts
sensitivity_results = []

for photon_level in noise_levels:
    # Generate noisy image
    scaled = clean_lf_image * photon_level / (clean_lf_image.max() + 1e-10)
    noisy = np.random.poisson(scaled).astype(np.float32)
    noisy = noisy / photon_level * clean_lf_image.max()
    noisy = np.clip(noisy, 0, None)
    
    # Quick reconstruction (fewer iterations for speed)
    recon, errors, _ = richardson_lucy_lfm(
        observed_image=noisy,
        psf_sigmas=PSF_SIGMAS,
        lenslet_pattern=lenslet_pattern,
        volume_shape=VOLUME_SHAPE,
        n_iterations=30,
        filter_flag=True,
        lanczos_window=3,
        verbose=False
    )
    
    # Compute metrics
    psnr = compute_psnr(normalize_for_display(ground_truth_volume), 
                        normalize_for_display(recon), data_range=1.0)
    ncc = compute_ncc(ground_truth_volume, recon)
    
    sensitivity_results.append({
        'photon_level': photon_level,
        'PSNR': psnr,
        'NCC': ncc,
        'final_error': errors[-1]
    })
    print(f"  Photon level {photon_level:5d}: PSNR = {psnr:.2f} dB, NCC = {ncc:.4f}")

# Plot sensitivity results
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

photon_levels = [r['photon_level'] for r in sensitivity_results]
psnr_vals = [r['PSNR'] for r in sensitivity_results]
ncc_vals = [r['NCC'] for r in sensitivity_results]
error_vals = [r['final_error'] for r in sensitivity_results]

axes[0].semilogx(photon_levels, psnr_vals, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Photon Level')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_title('Reconstruction Quality vs Noise')
axes[0].grid(True, alpha=0.3)

axes[1].semilogx(photon_levels, ncc_vals, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Photon Level')
axes[1].set_ylabel('NCC')
axes[1].set_title('Correlation vs Noise')
axes[1].grid(True, alpha=0.3)

axes[2].loglog(photon_levels, error_vals, 'ro-', linewidth=2, markersize=8)
axes[2].set_xlabel('Photon Level')
axes[2].set_ylabel('Final Convergence Error')
axes[2].set_title('Convergence Error vs Noise')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Sensitivity Analysis: Effect of Photon Noise Level', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Discussion & Key Takeaways

### Summary of Findings

In this tutorial, we implemented Richardson-Lucy deconvolution for light field microscopy 3D reconstruction. Key observations:

1. **Successful 3D recovery**: The algorithm successfully recovers the 3D structure from a single 2D lenslet image, demonstrating the power of computational imaging.

2. **Depth-dependent quality**: Reconstruction quality varies with depth, with in-focus planes typically showing better fidelity than peripheral planes.

3. **Noise sensitivity**: Higher photon counts lead to significantly better reconstructions, highlighting the importance of signal-to-noise ratio in inverse problems.

4. **Convergence behavior**: The algorithm shows rapid initial convergence followed by gradual refinement, with the anti-aliasing filter providing important regularization.

### Limitations

1. **Simplified forward model**: Our tutorial uses a simplified PSF model; real LFM systems have more complex, spatially-varying PSFs.

2. **Computational cost**: Each iteration requires forward and backward projections through the entire volume, making the algorithm computationally intensive for large volumes.

3. **No explicit regularization**: Beyond the anti-aliasing filter, we don't use explicit priors (e.g., sparsity, total variation), which could improve results.

4. **Fixed iteration count**: Optimal stopping criteria (e.g., based on cross-validation) could improve results.

### Extensions and Improvements

1. **GPU acceleration**: Using CuPy or PyTorch for GPU-accelerated projections (as in the reference implementation)

2. **Wave optics PSF**: More accurate PSF models based on wave optics theory

3. **Regularized variants**: TV-regularized or sparse Richardson-Lucy algorithms

4. **Deep learning**: Neural network-based approaches for faster, higher-quality reconstruction

### Key References

1. Broxton, M., et al. "Wave optics theory and 3-D deconvolution for the light field microscope." *Optics Express* 21(21), 25418-25439 (2013)

2. Prevedel, R., et al. "Simultaneous whole-animal 3D imaging of neuronal activity using light-field microscopy." *Nature Methods* 11(7), 727-730 (2014)

3. Stefanoiu, A., et al. "Artifact-free deconvolution in light field microscopy." *Optics Express* 27(22), 31644-31666 (2019)

In [ ]:
# =============================================================================
# Cell 15: Summary Metrics Table
# =============================================================================

print("="*70)
print("                    RECONSTRUCTION SUMMARY REPORT")
print("="*70)
print()

# Simulation parameters
print("SIMULATION PARAMETERS")
print("-" * 70)
print(f"  {'Volume dimensions:':<30} {VOLUME_SHAPE[0]} x {VOLUME_SHAPE[1]} x {VOLUME_SHAPE[2]} (ny x nx x nz)")
print(f"  {'Lenslet pitch:':<30} {LENSLET_PITCH} pixels")
print(f"  {'Depth range:':<30} [{DEPTH_RANGE[0]:.1f}, {DEPTH_RANGE[-1]:.1f}] (normalized units)")
print(f"  {'Photon level:':<30} {PHOTON_LEVEL}")
print(f"  {'Number of iterations:':<30} {N_ITERATIONS}")
print(f"  {'Anti-aliasing filter:':<30} {'Enabled' if FILTER_FLAG else 'Disabled'}")
print()

# Overall metrics
print("OVERALL RECONSTRUCTION QUALITY")
print("-" * 70)
print(f"  {'Metric':<25} {'Value':<15} {'Interpretation'}")
print(f"  {'-'*25} {'-'*15} {'-'*25}")
print(f"  {'PSNR:':<25} {overall_psnr:>10.2f} dB   {'(Higher is better)'}")
print(f"  {'SSIM:':<25} {overall_ssim:>10.4f}      {'(1.0 = perfect)'}")
print(f"  {'NCC:':<25} {overall_ncc:>10.4f}      {'(1.0 = perfect)'}")
print(f"  {'Mean Absolute Error:':<25} {error_volume.mean():>10.4f}      {'(Lower is better)'}")
print()

# Per-depth metrics
print("PER-DEPTH RECONSTRUCTION QUALITY")
print("-" * 70)
print(f"  {'Depth':<8} {'z-index':<10} {'PSNR (dB)':<12} {'SSIM':<10} {'NCC':<10}")
print(f"  {'-'*8} {'-'*10} {'-'*12} {'-'*10} {'-'*10}")
for m in slice_metrics:
    print(f"  {m['depth']:>6.1f}   {m['z']:>6d}     {m['PSNR']:>8.2f}     {m['SSIM']:>8.4f}   {m['NCC']:>8.4f}")
print()

# Convergence statistics
print("CONVERGENCE STATISTICS")
print("-" * 70)
print(f"  {'Initial error:':<30} {error_metrics[0]:.6f}")
print(f"  {'Final error:':<30} {error_metrics[-1]:.6f}")
print(f"  {'Error reduction factor:':<30} {error_metrics[0]/error_metrics[-1]:.2f}x")
print(f"  {'Total computation time:':<30} {sum(iteration_times):.2f} seconds")
print(f"  {'Mean time per iteration:':<30} {np.mean(iteration_times):.4f} seconds")
print()

# Sensitivity analysis summary
print("NOISE SENSITIVITY SUMMARY")
print("-" * 70)
print(f"  {'Photon Level':<15} {'PSNR (dB)':<12} {'NCC':<10}")
print(f"  {'-'*15} {'-'*12} {'-'*10}")
for r in sensitivity_results:
    print(f"  {r['photon_level']:<15} {r['PSNR']:>8.2f}     {r['NCC']:>8.4f}")
print()

print("="*70)
print("                         END OF REPORT")
print("="*70)

## Conclusion

In this tutorial, we explored **Light Field Microscopy (LFM) 3D reconstruction** as a canonical inverse problem in computational imaging. We demonstrated:

### The Problem
- LFM captures 3D volumetric information in a single 2D snapshot using a microlens array
- Recovering the 3D volume from the 2D measurement is an ill-posed inverse problem due to dimensional reduction, depth-dependent blurring, and noise

### The Method
- **Richardson-Lucy deconvolution** provides an iterative maximum-likelihood solution under Poisson noise statistics
- The algorithm uses alternating forward and backward projections with multiplicative updates
- **Anti-aliasing filtering** (Lanczos) provides implicit regularization to suppress artifacts

### Key Results
- Successful 3D reconstruction from synthetic lenslet images with realistic noise
- Quantitative evaluation showing PSNR improvements and correlation with ground truth
- Sensitivity analysis demonstrating the importance of signal-to-noise ratio

### Broader Impact
Light field microscopy enables high-speed volumetric imaging critical for neuroscience (whole-brain calcium imaging), developmental biology, and cardiac research. The computational reconstruction techniques demonstrated here are fundamental to extracting meaningful 3D information from these measurements.

This tutorial provides a foundation for understanding more advanced LFM reconstruction methods, including GPU-accelerated implementations, wave-optics-based forward models, and deep learning approaches that are pushing the boundaries of what's possible in computational microscopy.